### 一些读写数据的基本方法

In [4]:
import os
import pickle as pkl
import numpy as np
import pandas as pd
import torch
import csv
from tqdm import trange
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.tsatools import detrend
from scipy.stats import normaltest, norm

root_pth = '/nas/datasets/Tensor-Time-Series-Dataset/Processed_Data'
output_csv_pth = '/home/wangzihan/workspace/TTS_results/csv'
output_img_pth = '/home/wangzihan/workspace/TTS_results/imgs'
dataset_list = [ 'JONAS_NYC_taxi', 'JONAS_NYC_bike',
                'ETT_hour',
                'PEMS03', 'PEMS07', 
                'METRO_SH', 'METRO_HZ',
                'electricity', 'weather', 'Jena_climate', 'nasdaq100']

In [5]:
class DataParserBase:
    def __init__(self, dataset_name:str, window_size, stride) -> None:
        # get data from pkl file 
        self.dataset_name = dataset_name
        pkl_name = dataset_name + '.pkl'
        pkl_path = os.path.join(root_pth, dataset_name, pkl_name)
        if not os.path.exists(pkl_path):
            raise FileExistsError(f"Can not find file: {pkl_path}")
        with open(pkl_path, 'rb') as file:
            self.data_pkl = pkl.load(file)
            self.data = self.data_pkl['data']
        # transform data to 2D MTS array
        self.transform_data()
        self.window_size = window_size
        self.stride = (1 if (window_size == 0) else stride)
        
    # treat data as a collection of 1D time series to btain statistics of channels
    # data dims: [timesteps, channel_nums]
    def transform_data(self):
        temp = self.data.reshape(self.data.shape[0], -1, 1)
        self.data = np.squeeze(temp, axis=-1)
        self.data_shape =self.data.shape
        self.timesteps = self.data_shape[0]
        self.channel_nums = self.data_shape[1]

    def get_data_shape(self):
        return self.data_shape

### Seasonality/Trend 强度

##### （1）对Tensor/MTS中的每个一维通道做如下操作：
1. 利用FFT谱分析计算时间序列的周期：取FFT谱幅度最大的一项作为序列的周期长度；
2. 以得到的周期为参数，利用STL将时间序列分为趋势、季节和残差三项；
    $$
    X = season + trend + resid
    $$

3. 利用分解结果计算seasonality和trend的强度，公式如下。
    $$
    strength_{season} = max(1 - \frac{var(resid)}{var(season + resid)}, 0)
    $$
    $$
    strength_{trend} = max(1 - \frac{var(resid)}{var(trend + resid)}, 0)
    $$
##### （2）计算所有通道的平均值和方差作为统计结果

In [4]:
class STParser(DataParserBase):
    def __init__(self, dataset_name:str, window_size, stride=0) -> None:
        super().__init__(dataset_name, window_size, stride)
    
    def get_period(self, ts, k=1):
        ts = detrend(ts, order=1)
        # 计算FFT
        if np.nonzero(ts)[0].shape[0] == 0:
            return np.ones(k), np.ones(k)
        fft = np.abs(np.fft.fft(ts, axis=0))
        frequencies = np.fft.fftfreq(len(ts))

        # 找到最大的k个频率
        indices = np.argsort(np.abs(fft[1:-1]))[-k:]
        periods = (1 / (frequencies[indices]+1e-10) if frequencies[indices] != 0 else np.array([1]))
        strength = fft[indices]/np.sum(fft)
        
        return np.abs(periods.astype(int))
    
    def measure_strength(self):
        trd = []
        ses = []
        for i in trange(0, self.channel_nums):
            if self.window_size != 0:
                j = 0
                while j < self.timesteps - self.window_size:
                    seg = self.data[j:j+self.window_size, i]
                    ser = (seg - np.mean(seg)) / np.std(seg) # z-score normalize
                    if np.all(ser == 0):
                        continue
                    else:
                        period = self.get_period(ser)[0]
                        if period != 1:
                            stl = STL(ser, period = int(period), robust = True).fit()
                        else:
                            stl = STL(ser, period = 7, robust = True).fit()
                        # seasonality and trend strength
                        seasonal, trend, resid = stl.seasonal, stl.trend, stl.resid
                        if (np.var(trend+resid) == 0) or (np.var(seasonal+resid) == 0):
                            j += self.stride
                            continue
                        else:
                            val_trd = 1- (np.var(resid)/np.var(trend+resid))
                            trd.append((val_trd if val_trd > 0 else 0))
                            val_ses = 1- (np.var(resid)/np.var(seasonal+resid))
                            ses.append((val_ses if val_ses > 0 else 0))
                            j += self.stride
            else:
                seg = self.data[:, i]
                ser = (seg - np.mean(seg)) / np.std(seg) # z-score normalize
                seq = self.data[:, i] # seq: sequence to decompose
                if np.std(seq) == 0 or np.std(seg) == 0: # ditch constants
                    continue
                else:
                    period = self.get_period(ser)[0] # calculate period in a sufficiently long segment
                    seq = (seq - np.mean(seq)) / np.std(seq) # normalize before decompose
                    if period != 1:
                        stl = STL(seq, period=int(period), robust=True).fit()
                    else:
                        stl = STL(seq, period=7, robust=True).fit()
                    # seasonality and trend strength
                    seasonal, trend, resid = stl.seasonal, stl.trend, stl.resid
                    if (np.var(trend+resid) == 0) or (np.var(seasonal+resid) == 0):
                        continue
                    else:
                        val_trd = 1- (np.var(resid)/np.var(trend+resid))
                        trd.append((val_trd if val_trd > 0 else 0)) # minimum 0
                        val_ses = 1- (np.var(resid)/np.var(seasonal+resid))
                        ses.append((val_ses if val_ses > 0 else 0)) # minimum 0

        return trd, ses
    
    def st_parse(self):
        all_trd = []
        all_ses = []
        print(f"Start parsing dataset: {self.dataset_name}")
        all_trd, all_ses = self.measure_strength()
        sm = np.mean(all_ses)
        sv = np.var(all_ses)
        tm = np.mean(all_trd)
        tv = np.var(all_trd)

        return sm, sv, tm, tv

In [ ]:
parser = STParser('METRO_HZ', window_size=0)
sm, sv, tm, tv = parser.st_parse()
print(f"Seasonality strength mean: {sm}, Seasonality variance: {sv}")
print(f"Trend strength mean: {tm}, Trend variance: {tv}")

### 结果
将所有数据集都算一遍需要很长时间，汇总成表如下。

In [3]:
tab =pd.read_csv(os.path.join(output_csv_pth, 'result_st.csv'))
print(tab.iloc[0:len(dataset_list), [0,1,3,2,4]])
print(f'window size: 0, stride: 1')

           dataset  season_mean  trend_mean  season_var  trend_var
0   JONAS_NYC_taxi     0.552178    0.271624    0.049679   0.032389
1   JONAS_NYC_bike     0.531289    0.224330    0.011175   0.006498
2         ETT_hour     0.469383    0.582339    0.126486   0.087390
3           PEMS03     0.852112    0.138501    0.003084   0.012041
4           PEMS07     0.831834    0.134290    0.006878   0.018255
5         METRO_SH     0.557811    0.122709    0.027951   0.011978
6         METRO_HZ     0.445159    0.102397    0.035750   0.012706
7      electricity     0.782912    0.569046    0.056194   0.035582
8          weather     0.212814    0.589898    0.084626   0.181028
9     Jena_climate     0.505811    0.105059    0.061897   0.043497
10       nasdaq100     0.297781    0.722626    0.106529   0.154577
window size: 0, stride: 1


### Normality和JS散度

##### (1) 对Tensor/MTS的每个一维通道做如下操作:
1. 将时间序列看作从分布中采样的一些样点，利用scipy.stats中的normaltest()方法对其分布是否为正态分布进行p-检验。原假设为“序列服从正态分布”，p<0.05时拒绝原假设。

2. 计算时间序列的均值和方差，生成同均值和方差的正态分布，用直方图近似作为二者的分布，计算两者的JS散度,计算方法如下。注意计算时取以2为底的对数，因此JS散度的取值在0和1之间，数值越大说明两个分布差别越大。
$$
JSDiv(P, Q) 
= \frac{1}{2} H(P, \frac{P+Q}{2}) + \frac{1}{2} H(Q, \frac{P+Q}{2})
$$
$$
= \frac{1}{2} \sum_{x}{P·log_{2}(\frac{2P}{P+Q})} + \frac{1}{2} \sum_{x}{Q·log_{2}(\frac{2Q}{P+Q})}
$$


##### (2) 统计所有通道正态检验的p值和JS散度的均值
##### (3) 统计每个数据集中正态分布通道占总通道的比例

In [8]:
class DistributionParser(DataParserBase):
    def __init__(self, dataset_name: str, window_size = 0, stride = 1) -> None:
        super().__init__(dataset_name, window_size, stride )
    
    def JS_divergence(self, p, q, eps=1e-10):
        m = 0.5 * (p + q)
        return 0.5 * (np.sum(p * np.log2((p+eps)/m)) + np.sum(q * np.log2((q+eps)/m)))
    
    def JS_div(self, arr1, arr2, num_bins):
        max0 = max(max(arr1), max(arr2))
        min0 = min(min(arr1), min(arr2))
        bins = np.linspace(min0, max0, num_bins)

        # get distribution for each 'stochastic' array
        pdf1 = pd.cut(arr1, bins=bins, duplicates='drop').value_counts()
        pdf2 = pd.cut(arr2, bins=bins, duplicates='drop').value_counts()

        if sum(pdf1) > 0 and sum(pdf2) > 0:
            pdf1 = pdf1 / len(arr1)
            pdf2 = pdf2 / len(arr2)
            return self.JS_divergence(pdf1, pdf2)
        else:
            return None

    def normal_test(self):
        score_list = []
        normal_count = 0
        for i in trange(self.channel_nums):
            if self.window_size == 0:
                seq = self.data[:, i]
                res = normaltest(seq)[1]
                if sum(seq) == 0:
                    continue
                score_list.append(res)
                if res > 0.05:
                    normal_count += 1
            else:
                j = 0
                pval = []
                while j < self.timesteps - self.window_size:
                    seg = self.data[j:j+self.window_size, i]
                    ser = (seg - np.mean(seg)) / np.std(seg) # z-score normalize
                    if np.all(ser == 0):
                        continue
                    else:
                        res = normaltest(ser)[1]
                        pval.append(res)
                    j += self.stride
                res = np.mean(pval)
                score_list.append(res)
                if res > 0.05:
                    normal_count += 1
        
        score_mean = np.mean(score_list)
        normal_rate = normal_count / self.channel_nums

        return score_mean, normal_rate
    
    def JS_Div_strength(self):
        js_list = []
        for i in trange(self.channel_nums):
            all_mean = np.mean(self.data[:, i])
            all_std = np.std(self.data[:, i])
            if all_std == 0:
                seq = self.data[:, i] - all_mean
            else:
                seq = (self.data[:, i] - all_mean) / all_std
            if sum(seq) == 0:
                continue

            if self.window_size == 0:
                norm_distrib = norm.rvs(loc=all_mean, scale=all_std, size=len(seq))
                js = self.JS_div(self.data[:, i], norm_distrib, 20)
                if js is not None:
                    js_list.append(js)
            else:
                j = 0
                js_seq_list = []
                while j < self.timesteps - self.window_size:
                    seg = self.data[j:j+self.window_size, i]
                    mean = np.mean(seg)
                    std = np.std(seg)
                    if std == 0:
                        continue
                    else:
                        norm_distrib = norm.rvs(loc=mean, scale=std, size=len(seg))
                        js = self.JS_div(seg, norm_distrib, 20)
                        if js is not None:
                            js_seq_list.append(js)
                    js = np.mean(js_seq_list)
                    j += self.stride
                js_list.append(js)

        js_mean = np.mean(js_list)

        return js_mean
    
    def distrib_parse(self):
        print(f"Start parsing dataset: {self.dataset_name}")
        score_mean, normal_rate = self.normal_test()
        js_mean = self.JS_Div_strength()
        return score_mean, normal_rate, js_mean

In [ ]:
parser = DistributionParser('ETT_hour', window_size=336, stride=336)
score_mean, normal_rate, js_mean = parser.distrib_parse()
print(f"JS divergence mean: {js_mean}")

Start parsing dataset: ETT_hour


100%|██████████| 7/7 [00:01<00:00,  3.60it/s]

JS divergence mean: (0.05214285274415403, 0.5714285714285714, 0.07271570079378784)


### 结果

In [9]:
tab =pd.read_csv(os.path.join(output_csv_pth, 'dataset_stats_1.csv'))
print(tab.iloc[0:len(dataset_list), :])
print(f'window_size: {0}, stride: {1}')

      dataset_name  normaltest_p_mean  normal_rate  JS_Div_mean
0   JONAS_NYC_taxi       1.571480e-08     0.000000     0.154981
1   JONAS_NYC_bike      2.051336e-139     0.000000     0.265742
2         ETT_hour       2.611270e-25     0.000000     0.057880
3           PEMS03       8.937862e-31     0.000000     0.093441
4           PEMS07       6.392552e-10     0.000000     0.103641
5         METRO_SH       2.695256e-16     0.000000     0.207051
6         METRO_HZ       1.935408e-11     0.000000     0.147724
7      electricity       2.469007e-03     0.003115     0.092365
8          weather      5.313537e-194     0.000000     0.158067
9     Jena_climate      2.792172e-115     0.000000     0.043970
10       nasdaq100      6.365130e-135     0.000000     0.146511
window_size: 0, stride: 1


### 平稳性

参考TFB: Towards Comprehensive and Fair Benchmarking of Time
Series Forecasting Methods (PVLDB 2024) 

(https://arxiv.org/abs/2403.20150)

##### （1）对每个通道做以下操作
1. 进行Augmented Dickey-Fuller Test（利用statsmodels库中的adfuller()方法）。原理是拟合时间序列，统计如下的AR模型的参数γ。γ=0对应随机游走模型（不平稳），γ<0则平稳。对得到的参数进行p-检验，原假设为时间序列不平稳（γ=0），p<0.05时拒绝原假设。
$$
{\Delta}y_{t} = {\alpha} + {\beta}t + {\gamma}y_{t-1} + {\delta}_{1}{\Delta}y_{t-1} + {\delta}_{2}{\Delta}y_{t-2}...
$$

2. 统计时取β=0，检测序列是否符合带偏移的随机游走。根据p值统计平稳的通道数。

In [ ]:
from statsmodels.tsa.stattools import adfuller
class StationaryParser(DataParserBase):
    def __init__(self, dataset_name: str, window_size = 0, stride = 1) -> None:
        super().__init__(dataset_name, window_size, stride)
    
    def measure_stationarity(self):
        if self.window_size == 0:
            station_count = 0
            adf_pval = []

            # Null hypothesis: unit root exists -> "non-stationary"
            # at p-value < 0.05, reject null hypothesis
            for i in trange(self.channel_nums):
                seq = self.data[:, i]
                if np.std(seq) == 0:
                    continue
                # random walk with drift
                else:
                    res = adfuller(seq, regression='c')[1]
                    if res < 0.05:
                        station_count += 1
                    adf_pval.append(res)
            
            p_mean = np.mean(adf_pval)
            station_rate = station_count / self.channel_nums

            return p_mean, station_rate

### 结果

In [6]:
tab = pd.read_csv(os.path.join(output_csv_pth, 'result_stationary.csv'))
print(tab.iloc[0:len(dataset_list), :])
print(f'window size: 0, stride: 1')

      dataset_name    ADF_p_mean  stationary_rate
0   JONAS_NYC_taxi  5.567334e-14         1.000000
1   JONAS_NYC_bike  4.652167e-15         0.773438
2         ETT_hour  1.199963e-03         1.000000
3           PEMS03  2.260248e-29         1.000000
4           PEMS07  5.144269e-20         1.000000
5         METRO_SH  1.167720e-09         1.000000
6         METRO_HZ  2.379435e-12         1.000000
7      electricity  5.149945e-03         0.981308
8          weather  1.036509e-08         1.000000
9     Jena_climate  5.233535e-17         1.000000
10       nasdaq100  4.654827e-01         0.254369
window size: 0, stride: 1


#### $r_{1}$和$r_{2}$
参考BasicTS+（https://arxiv.org/abs/2310.06119）。

BasicTS提出了r<sub>1</sub>和r<sub>2</sub>两个指标，用于衡量MTS时间序列数据过去和未来的相似性，使用内积刻画两个序列之间的相似程度。

具体指标计算方法如下，待选取的超参数有lookback window的长度T<sub>p</sub>，lookahead window的长度T<sub>f</sub>以及两个阈值e<sub>u</sub>和e<sub>l</sub>。X<sup>i</sup><sub>(t-T<sub>p</sub>:t)</sub>表示MTS时间序列第i维特征，时间范围t-T<sub>p</sub>到t的一段，其余表记类似。I()为指示函数，用于统计满足条件的点。

$$r_{1} = \frac{\sum_{t,i,j}I(A_{t,i,j}^{p}>e_{u} \wedge A_{t,i,j}^{p}<e_{l})}{T·N·N}$$
$$r_{2} = \frac{\sum_{t,i,j}I(A_{t,i,j}^{p}>e_{u} \wedge A_{t,i,j}^{p}<e_{l})}{\sum_{t,i,j}I(A_{t,i,j}^{p}>e_{u})}$$

其中：

$$A_{t,i,j}^{p} = \frac{X_{[t-T_{p}:t]}^{i} · X_{[t-T_{p}:t]}^{i}}{||X_{[t-T_{p}:t]}^{i}|| · ||X_{[t-T_{p}:t]}^{j}||}$$

$$A_{t,i,j}^{f} = \frac{X_{[t:t+T_{f}]}^{i} · X_{[t:t+T_{f}]}^{i}}{||X_{[t:t+T_{f}]}^{i}|| · ||X_{[t:t+T_{f}]}^{j}||}$$

上述A<sub>p</sub>和A<sub>f</sub>是两个TxNxN维Tensor，A<sub>p</sub>[t, i, j]表示在t时间点，第i维和第j维特征之间，lookback window内的数据相似程度；A<sub>t</sub>[t, i, j]类似，表示lookahead window内的数据相似程度。构建两个矩阵后，再通过统计满足阈值条件的点计算r<sub>1</sub>和r<sub>2</sub>。

直观地，r<sub>1</sub>表示“lookback数据相似但待预测数据不相似”情形在全体数据中出现的频繁程度，r<sub>2</sub>表示“lookback数据相似但待预测数据不相似”情形在所有“lookback数据相似”数据中出现的频繁程度。

In [ ]:
class HeteroParser(DataParserBase):
    def __init__(self, dataset_name: str, window_size = 0, stride = 1) -> None:
        super().__init__(dataset_name, window_size, stride)
    
    def get_Ap_Af(self, hist=12, pred=12):
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        # "drop last"
        Ap = torch.zeros(self.timesteps-hist-pred, self.channel_nums, self.channel_nums).to(device)
        Af = torch.zeros(self.timesteps-hist-pred, self.channel_nums, self.channel_nums).to(device)

        for i in range(self.channel_nums):
            for j in range(self.channel_nums):
                # calculate A_p
                h_i = [self.data[t-hist:t, i] for t in range(hist, self.timesteps-pred)]
                h_i = torch.cat(h_i, dim=0).view(-1, hist) # shape: [timesteps-hist-pred, hist]
                h_j = [self.data[t-hist:t, j] for t in range(hist, self.timesteps-pred)]
                h_j = torch.cat(h_j, dim=0).view(-1, hist)
                Ap[:, i, j] = torch.sum(h_i * h_j, dim=-1) / torch.sum(h_i*h_i, dim=-1)

                # calculate A_t
                p_i = [self.data[t:t+pred, i] for t in range(hist, self.timesteps-pred)]
                p_i = torch.cat(p_i, dim=0).view(-1, pred)
                p_j = [self.data[t:t+pred, j] for t in range(hist, self.timesteps-pred)]
                p_j = torch.cat(p_j, dim=0).view(-1, pred)
                Af[:, i, j] = torch.sum(p_i * p_j, dim=-1) / torch.sum(p_i*p_i, dim=-1)
        
        return Ap, Af
    
    def measure_r1_r2(self, e_u=0.9,e_l=0.5):
        Ap, Af = self.get_Ap_Af()
        T = self.timesteps - self.hist - self.pred
        N = self.channel_nums
        r1 = torch.sum((Ap > e_u).to(dtype=int) & (Af < e_l).to(dtype=int)).cpu().numpy() / (T * N * N)
        r2 = torch.sum((Ap > e_u).to(dtype=int) & (Af < e_l).to(dtype=int)).cpu().numpy() / torch.sum((Ap > e_u).to(dtype=int)).cpu().numpy()
        r1 = r1.tolist()
        r2 = r2.tolist()

        return r1, r2


#### Hurst和lumpiness